In [126]:
from sympy import symbols, Rational
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
from codegen.codegen_utils import * 
init_printing()
printer = MyPrinter() 


In [132]:
x,y,z = symbols("x y z", real=True)
r = sp.sqrt(x**2+y**2+z**2)

B = [x/r**3,y/r**3,z/r**3]

sp.simplify(sp.diff(B[0],x) + sp.diff(B[1],y) + sp.diff(B[2],z))

In [127]:
M, rc, Tc, uc, gamma, n, K, T, r= symbols("M rc Tc uc gamma n K T r")

scalar = ("double", None)
ABI = {
    "M": scalar,
    "rc": scalar,
    "gamma": scalar,
    "n": scalar,
    "K": scalar,
    "T": scalar,
    "r": scalar,
    "Tc": scalar,
    "uc": scalar
}

In [128]:
u1_c = - sp.sqrt(M/(2*rc))
T_c = n/(n+1) * (u1_c**2)/(1-(n+3)*u1_c**2) 

In [129]:
C1 = Tc**n * uc * rc**2 
C2 = (1+(n+1)*Tc)**2*(1-2*M/rc + uc**2)

In [130]:
f = sp.simplify((1+(n+1)*T)**2*(1-2*M/r + C1**2/(r**4*T**(2*n))) - C2)

In [108]:
u1 = C1/(r**2*T**n)
rho = (T/K)**n 
p = T * rho 

Now we want to add a small angular momentum parametrized by 
$$
j = -\frac{u_\phi}{u_t}
$$
In BL coordinates we have: 
$$ 
u_r = g_{r r} u^r
$$
Which is known. 
Moreover, we can write (using $u^\theta = 0$):
$$
- 1 = u^\mu u_\mu = g^{tt} (u_t)^2 + g^{rr} (u_r)^2 + g^{\phi \phi} (u_\phi)^2 + 2 g^{\phi t} u_\phi u_t 
$$
Using the definition of $j$ 
$$
-1 = g^{rr} (u_r)^2 - 2 g^{t \phi} ((u_\phi)^2 / j) + g^{\phi\phi} (u_\phi)^2 + g^{tt} (u_\phi/j)^2
$$
So that
From which we can also find $u_t$ and raise the indices to find $u^\mu = (u^t, u^r, 0, u^\phi)$.


In [109]:
flist = []

out_list = ["dT"]
out_abi = {"dT": scalar}
flist.append(make_function([f],printer,"bondi_T__r",ABI,out_list,out_abi))

out_list = ["ur","rho","press"]
out_abi = {"ur": ("double",None), "rho": ("double",None), "press": ("double",None)}
flist.append(make_function([u1,rho,p],printer,"bondi_ur_rho_p__r",ABI,out_list,out_abi))


out_list = ["ur_c","T_c"]
out_abi = {"ur_c": ("double",None), "T_c": ("double",None)}
flist.append(make_function([u1_c,T_c],printer,"bondi_uc_Tc",ABI,out_list,out_abi))

printed_functions = '\n' + '\n'.join(flist)

In [110]:
file = '''
/****************************************************************************/
/*                     Bondi ID helpers, SymPy generated                    */
/****************************************************************************/
#ifndef GRACE_BONDI_ID_SUBEXPR_HH
#define GRACE_BONDI_ID_SUBEXPR_HH

#include <Kokkos_Core.hpp>
''' + printed_functions + '''
#endif 
'''
with open("bondi_subexpressions.hh","w") as ff:
    ff.write(file)

In [111]:
f.subs({M:1, n:3, rc:8, Tc:0.075, uc: -0.25, r: 7.979946872679769, T: 0.07500420039979833})

In [112]:
T__r = sp.lambdify([rc,gamma,n,K,M,T,r,uc,Tc],f)
dT__r = sp.lambdify([rc,gamma,n,K,M,T,r,uc,Tc],df_dT)

T_c__l = sp.lambdify([rc,gamma,n,K,M],T_c)
u_c__l = sp.lambdify([rc,gamma,n,K,M],u1_c)

In [113]:
Gamma = 4/3 
NN = 1/((Gamma-1))
KK = 1
RC = 8 

In [114]:
TC = T_c__l(RC,Gamma,NN,KK,1)
UC = u_c__l(RC,Gamma,NN,KK,1)
AS = 1/(2*RC-3)
print(AS)

0.07692307692307693


In [115]:
from scipy.optimize import newton 
import numpy as np 
rotfun = lambda T,r: T__r(RC,Gamma,NN,K,1,T,r,UC,TC) 
drotfun = lambda T,r: dT__r(RC,Gamma,NN,K,1,T,r,UC,TC) 

In [116]:
def get_temp_min(r,t_min,t_max):
    ratio = 0.3819660112501051
    max_iter = 40 

    t_mid = t_min + ratio * (t_max-t_min)
    res_mid = rotfun(t_mid,r)
    larger_to_right = True 

    for i in range(max_iter):
        if res_mid < 0:
            return t_mid 

        if larger_to_right:
            t_new = t_mid + ratio * (t_max-t_mid) 

            res_new = rotfun(t_new,r)
            if ( res_new < res_mid ) :
                t_min = t_mid 
                t_mid = t_new 
                res_mid = res_new 
            else:
                t_max = t_new 
                larger_to_right = False 
        else:
            t_new = t_mid - ratio * (t_mid-t_min)
            res_new = rotfun(t_new,r)
            if ( res_new < res_mid ) :
                t_max = t_mid 
                t_mid = t_new 
                res_mid = res_new 
            else:
                t_min = t_new 
                larger_to_right = True 

         


In [118]:
from scipy.optimize import bisect 

def find_temp(r,t_min,t_max):
    tres = get_temp_min(r,t_min,t_max)
    print(tres)
    if ( r < RC ):
        return bisect(rotfun,t_min,tres,args=(r))
    else:
        return bisect(rotfun,tres,t_max,args=(r))

In [119]:
print(get_temp_min(2.1,0.01,10))

0.9107977380572476


In [120]:
TT = [] 
radii = np.linspace(0.1,5,100)

for rr in radii:
    t = find_temp(rr,1e-2,10)
    TT.append(t)
TT = np.array(TT)

3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
3.8258404523885496
1.4675213571656511
0.9107977380572476
0.9107977380572476
0.5667236191084037
0.5667236191084037
0.5667236191084037
0.354074118948844
0.354074118948844
0.354074118948844
0.354074118948844
0.354074118948844
0.354074118948844
0.22264950015955987
0.222649500159559

In [122]:
TT = [] 
RR = [] 
# Solve down 
radii_m = np.linspace(0.1,7.5,100)
for i,rr in enumerate(radii_m[::-1]):
    Tguess = TT[-1]*(1+1e-6) if i>0 else TC*(1+1e-6)
    TT.append(newton(rotfun,Tguess,args=[rr]))
    RR.append(rr)
TT = TT[::-1]
RR = RR[::-1]
# Solve up 
radii_p = np.linspace(8.5,10,100)
for i,rr in enumerate(radii_p):
    Tguess = TT[-1]*(1-1e-6) if i>0 else TC*(1-1e-6)
    TT.append(newton(rotfun,Tguess,args=[rr]))
    RR.append(rr)

TT = np.array(TT)
RR = np.array(RR)